In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score,precision_recall_curve, auc, precision_score, recall_score, confusion_matrix

import joblib

In [2]:
# Load the dataset
df= pd.read_csv("credit_train.csv")

In [3]:
#preprocessing the data
df = df.dropna(subset=["Loan Status"])
df = df.drop(columns=["Loan ID", "Customer ID"])

#encoding categorical variables
df["Loan Status"] = df["Loan Status"].map({
    "Fully Paid": 1,
    "Charged Off": 0
})

df["Years in current job"] = df["Years in current job"].replace({
    "< 1 year": 0, "1 year": 1, "2 years": 2,
    "3 years": 3, "4 years": 4, "5 years": 5,
    "6 years": 6, "7 years": 7, "8 years": 8,
    "9 years": 9, "10+ years": 10
})

df = df.drop(columns=["Months since last delinquent"])

# Define features and target variable
X = df.drop("Loan Status", axis=1)
y = df["Loan Status"]

# Identify numeric and categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])
 
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
 
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])


C:\Users\Sakthi\AppData\Local\Temp\ipykernel_8716\2931158334.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Years in current job"] = df["Years in current job"].replace({


In [4]:
# ── 5. Split ───────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Class balance — 0 (Charged Off): {(y_train==0).sum()} | 1 (Fully Paid): {(y_train==1).sum()}\n")

Train: 80000 rows | Test: 20000 rows
Class balance — 0 (Charged Off): 18111 | 1 (Fully Paid): 61889



In [5]:
# ── 6. Model with class_weight='balanced' ──────────────────────────────────
# WHY THIS FIXES PREDICT-ALL-1:
#   SMOTE created so many synthetic class-0 samples that the model
#   became confused and defaulted to always predicting 1.
#   class_weight='balanced' instead tells RandomForest internally to
#   penalise misclassifying the minority class (0) more heavily —
#   clean, effective, no synthetic data needed.
 
rf = RandomForestClassifier(
    n_estimators=100,        # 100 trees keeps file size small
    max_depth=12,            # enough depth to learn real patterns
    min_samples_leaf=10,     # prevents overfitting to majority class
    max_features="sqrt",     # standard for classification
    class_weight="balanced", # THE core fix — replaces SMOTE
    random_state=42,
    n_jobs=-1
)
 
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", rf)
])
 
print("Training... (this takes ~30-60 seconds)")
full_pipeline.fit(X_train, y_train)
print("Done.\n")

Training... (this takes ~30-60 seconds)
Done.



In [6]:
# ── 7. Default threshold results ───────────────────────────────────────────
y_prob = full_pipeline.predict_proba(X_test)[:, 1]
y_pred = full_pipeline.predict(X_test)
 
print("=" * 58)
print("RESULTS AT DEFAULT THRESHOLD (0.50)")
print("=" * 58)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}\n")
print(classification_report(
    y_test, y_pred,
    target_names=["Charged Off (0)", "Fully Paid (1)"]
))
 


RESULTS AT DEFAULT THRESHOLD (0.50)
Accuracy : 0.7202
ROC-AUC  : 0.7583

                 precision    recall  f1-score   support

Charged Off (0)       0.42      0.58      0.48      4528
 Fully Paid (1)       0.86      0.76      0.81     15472

       accuracy                           0.72     20000
      macro avg       0.64      0.67      0.65     20000
   weighted avg       0.76      0.72      0.73     20000



In [7]:
# ── 8. Threshold tuning ────────────────────────────────────────────────────
# Scan thresholds to find where both class precisions are highest
print("=" * 58)
print("FINDING BEST THRESHOLD...")
print("=" * 58)
 
best_thresh = 0.5
best_avg_precision = 0
for t in np.arange(0.25, 0.80, 0.01):

    yp = (y_prob >= t).astype(int)

    p0 = precision_score(y_test, yp, pos_label=0, zero_division=0)

    p1 = precision_score(y_test, yp, pos_label=1, zero_division=0)

    r0 = recall_score(y_test, yp, pos_label=0, zero_division=0)

    r1 = recall_score(y_test, yp, pos_label=1, zero_division=0)

    # Target: both precisions >= 0.65, maximise their average

    if p0 >= 0.62 and p1 >= 0.70 and r0 > 0.1 and r1 > 0.1:

        avg = (p0 + p1) / 2

        if avg > best_avg_precision:

            best_avg_precision = avg

            best_thresh = t

 

print(f"Best threshold: {best_thresh:.2f}\n")

 

y_pred_tuned = (y_prob >= best_thresh).astype(int)

print(f"RESULTS AT TUNED THRESHOLD ({best_thresh:.2f})")

print("-" * 58)

print(f"Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}")

print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}\n")

print(classification_report(

    y_test, y_pred_tuned,

    target_names=["Charged Off (0)", "Fully Paid (1)"]

))

 

cm = confusion_matrix(y_test, y_pred_tuned)

tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix:")

print(f"  True Negatives  (correctly rejected): {tn}")

print(f"  False Positives (wrongly approved)  : {fp}")

print(f"  False Negatives (wrongly rejected)  : {fn}")

print(f"  True Positives  (correctly approved): {tp}\n")

 

# ── 9. Save ────────────────────────────────────────────────────────────────

joblib.dump(full_pipeline, "model.joblib",   compress=9)

joblib.dump(float(best_thresh), "threshold.joblib", compress=9)

 

size_mb = os.path.getsize("model.joblib") / (1024 * 1024)

print(f"model.joblib    saved → {size_mb:.1f} MB")

print(f"threshold.joblib saved → {os.path.getsize('threshold.joblib') / 1024:.1f} KB")

 

if size_mb < 100:

    print(f"\n✅ Under GitHub 100MB limit — safe to push.")

else:

    print(f"\n⚠️  Over 100MB — reduce n_estimators to 50 and re-run.")

 

print("\n✅ Re-deploy app.py to Render. Done!")

 

print(df.columns)    

print(X.columns.tolist())



FINDING BEST THRESHOLD...
Best threshold: 0.25

RESULTS AT TUNED THRESHOLD (0.25)
----------------------------------------------------------
Accuracy : 0.8205
ROC-AUC  : 0.7583

                 precision    recall  f1-score   support

Charged Off (0)       1.00      0.21      0.34      4528
 Fully Paid (1)       0.81      1.00      0.90     15472

       accuracy                           0.82     20000
      macro avg       0.91      0.60      0.62     20000
   weighted avg       0.85      0.82      0.77     20000

Confusion Matrix:
  True Negatives  (correctly rejected): 937
  False Positives (wrongly approved)  : 3591
  False Negatives (wrongly rejected)  : 0
  True Positives  (correctly approved): 15472



NameError: name 'os' is not defined

In [ ]:
print(df.columns)    
print(X.columns.tolist())